# Cuaderno 6 - Combinar condiciones

## Antes de comenzar

En el cuaderno anterior utilizamos `filter()` con una sola condición por vez. Ahora vamos a construir reglas más completas combinando expresiones booleanas.

Trabajaremos con:

```text
&    AND
|    OR
~    NOT
```

La idea principal será **ver primero qué produce cada condición por separado** y luego observar qué ocurre cuando las combinamos. De esta manera podremos relacionar la lógica escrita en el código con las filas que finalmente permanecen en el DataFrame.

Seguiremos utilizando el conjunto de ventas de la tienda escolar para que cada resultado pueda interpretarse con facilidad.


## 1. Preparar el entorno y los datos

Instalamos Polars en Colab, importamos la biblioteca y reconstruimos el DataFrame que venimos utilizando.


In [22]:
!pip install -q polars

In [23]:
import polars as pl

In [24]:
datos = {
    "producto": ["Cuaderno", "Lapicera", "Mochila", "Calculadora", "Regla", "Carpeta"],
    "categoria": ["Librería", "Librería", "Accesorios", "Tecnología", "Librería", "Librería"],
    "precio": [3500, 1200, 28500, 18500, 900, 4200],
    "cantidad": [4, 10, 2, 3, 6, 5],
}

df = pl.DataFrame(datos)
df

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


El DataFrame ejecutado tiene `shape: (6, 4)` y contiene las columnas `producto`, `categoria`, `precio` y `cantidad`. Sobre estas seis filas construiremos las distintas combinaciones lógicas.

## 2. Observar dos condiciones por separado

Comenzamos con dos reglas:

- el producto pertenece a `Librería`;
- el precio es mayor que `2000`.

Las observamos juntas dentro de `select()`.


In [25]:
df.select(
    "producto",
    (pl.col("categoria") == "Librería").alias("es_libreria"),
    (pl.col("precio") > 2000).alias("precio_mayor_2000"),
)

producto,es_libreria,precio_mayor_2000
str,bool,bool
"""Cuaderno""",true,true
"""Lapicera""",true,false
"""Mochila""",false,true
"""Calculadora""",false,true
"""Regla""",true,false
"""Carpeta""",true,true


La salida tiene `shape: (6, 3)` y muestra, para cada producto, el resultado de las dos condiciones por separado. `Cuaderno` y `Carpeta` tienen `true` en ambas columnas; `Lapicera` y `Regla` cumplen solamente `es_libreria`; `Mochila` y `Calculadora` cumplen solamente `precio_mayor_2000`.

Esta tabla deja visible la lógica antes de aplicar ningún filtro.

## 3. Combinar con AND

Queremos ahora conservar únicamente los registros donde **las dos condiciones sean verdaderas al mismo tiempo**.


In [26]:
df.filter(
    (pl.col("categoria") == "Librería")
    & (pl.col("precio") > 2000)
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Carpeta""","""Librería""",4200,5


La salida tiene `shape: (2, 4)` y conserva únicamente `Cuaderno` y `Carpeta`. Son las dos filas en las que `es_libreria` y `precio_mayor_2000` eran verdaderas al mismo tiempo.

`Lapicera` y `Regla` pertenecen a Librería, pero no superan `2000`; `Mochila` y `Calculadora` superan ese precio, pero pertenecen a otras categorías.

## 4. Ver el resultado lógico de AND

También podemos materializar la combinación como una columna booleana.


In [27]:
df.select(
    "producto",
    (
        (pl.col("categoria") == "Librería")
        & (pl.col("precio") > 2000)
    ).alias("cumple_ambas"),
)

producto,cumple_ambas
str,bool
"""Cuaderno""",true
"""Lapicera""",false
"""Mochila""",false
"""Calculadora""",false
"""Regla""",false
"""Carpeta""",true


La salida tiene `shape: (6, 2)`. La columna `cumple_ambas` vale `true` solamente para `Cuaderno` y `Carpeta`, y `false` para las otras cuatro filas.

Esto permite ver directamente el efecto lógico de AND antes de utilizarlo para descartar registros.

## 5. Los paréntesis como parte de la lógica

En las expresiones anteriores cada comparación aparece encerrada entre paréntesis:

```python
(pl.col("categoria") == "Librería")
&
(pl.col("precio") > 2000)
```

Esta forma evita ambigüedades y hace visible qué condiciones estamos combinando. A partir de ahora mantendremos este criterio cada vez que usemos `&` o `|`.


## 6. Guardar condiciones con nombres

Podemos construir cada regla por separado y darles nombres significativos.


In [28]:
es_libreria = pl.col("categoria") == "Librería"
precio_mayor_2000 = pl.col("precio") > 2000

df.filter(
    es_libreria & precio_mayor_2000
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Carpeta""","""Librería""",4200,5


La salida vuelve a ser `shape: (2, 4)` y contiene exactamente `Cuaderno` y `Carpeta`. Separar las reglas en `es_libreria` y `precio_mayor_2000` no modifica el resultado; mejora la lectura y permite reutilizar esas expresiones.

## 7. AND sobre una misma columna

También podemos combinar condiciones numéricas.

Busquemos precios entre `1000` y `5000`, escribiendo el intervalo explícitamente con dos comparaciones.


In [29]:
df.filter(
    (pl.col("precio") >= 1000)
    & (pl.col("precio") <= 5000)
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Carpeta""","""Librería""",4200,5


La salida tiene tres filas: `Cuaderno` (`3500`), `Lapicera` (`1200`) y `Carpeta` (`4200`). Son los valores que cumplen simultáneamente `precio >= 1000` y `precio <= 5000`.

En el cuaderno anterior resolvimos la misma idea con `is_between()`. Aquí usamos dos comparaciones para observar explícitamente el funcionamiento de AND.

## 8. Combinar con OR

Ahora cambiaremos la pregunta. Queremos conservar productos que pertenezcan a `Tecnología` **o** a `Accesorios`.


In [30]:
df.filter(
    (pl.col("categoria") == "Tecnología")
    | (pl.col("categoria") == "Accesorios")
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3


La salida tiene `shape: (2, 4)` y conserva `Mochila` y `Calculadora`. `Mochila` cumple la condición `Accesorios` y `Calculadora` la condición `Tecnología`; con OR alcanza con que una de las dos sea verdadera.

## 9. Ver el OR antes de filtrar

Podemos observar también la combinación OR como una columna booleana.


In [31]:
df.select(
    "producto",
    (
        (pl.col("categoria") == "Tecnología")
        | (pl.col("categoria") == "Accesorios")
    ).alias("categoria_interes"),
)

producto,categoria_interes
str,bool
"""Cuaderno""",false
"""Lapicera""",false
"""Mochila""",true
"""Calculadora""",true
"""Regla""",false
"""Carpeta""",false


La salida tiene seis filas y una columna booleana adicional. `categoria_interes` es `true` únicamente para `Mochila` y `Calculadora` y `false` para los otros cuatro productos.

La misma expresión que aquí observamos como columna es la que, dentro de `filter()`, selecciona esas dos filas.

## 10. AND y OR responden preguntas diferentes

Comparemos dos expresiones parecidas:

```python
(pl.col("categoria") == "Librería")
& (pl.col("precio") > 2000)
```

y:

```python
(pl.col("categoria") == "Librería")
| (pl.col("precio") > 2000)
```

La primera exige que ambas condiciones se cumplan. La segunda conserva cualquier fila que cumpla al menos una.

Ejecutemos la segunda para observar la diferencia.


In [32]:
df.filter(
    (pl.col("categoria") == "Librería")
    | (pl.col("precio") > 2000)
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


La salida tiene `shape: (6, 4)`: en este caso permanecen las seis filas. Los cuatro productos de Librería cumplen la primera condición, mientras que `Mochila` y `Calculadora` no son de Librería pero sí tienen un precio mayor que `2000`.

El contraste con AND es claro: cambiar `&` por `|` hizo que la condición dejara de ser restrictiva para este conjunto de datos.

## 11. OR frente a `is_in()`

Cuando varias alternativas pertenecen a una misma columna, `is_in()` puede expresar la intención de forma más compacta.


In [33]:
df.filter(
    pl.col("categoria").is_in(
        ["Librería", "Tecnología"]
    )
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Calculadora""","""Tecnología""",18500,3
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


La salida tiene cinco filas: los cuatro productos de `Librería` y `Calculadora`, que pertenece a `Tecnología`. Solo `Mochila`, cuya categoría es `Accesorios`, queda fuera.

Podríamos obtener este mismo resultado con dos comparaciones unidas mediante `|`, pero `is_in()` expresa de forma más directa que aceptamos cualquiera de varios valores posibles.

## 12. Negar una condición con NOT

El operador `~` invierte el resultado booleano de una expresión.

Partimos de la condición:

```python
pl.col("categoria") == "Librería"
```

y negamos su resultado.


In [34]:
df.filter(
    ~(pl.col("categoria") == "Librería")
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3


La salida tiene dos filas: `Mochila` y `Calculadora`. Son precisamente los registros que **no** pertenecen a Librería.

En este caso `!= "Librería"` sería una forma más directa de expresar la misma regla, pero `~` será más útil cuando neguemos una expresión compuesta.

## 13. Negar una combinación completa

Construimos ahora una regla compuesta:

```text
es Librería AND precio > 2000
```

y negamos todo el bloque.


In [35]:
df.filter(
    ~(
        (pl.col("categoria") == "Librería")
        & (pl.col("precio") > 2000)
    )
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Regla""","""Librería""",900,6


La salida contiene cuatro filas: `Lapicera`, `Mochila`, `Calculadora` y `Regla`. Quedaron excluidos únicamente `Cuaderno` y `Carpeta`, porque eran las dos filas que cumplían simultáneamente `Librería` y `precio > 2000`.

La negación se aplicó al bloque completo, no a una sola de sus partes.

## 14. Combinar tres condiciones

Podemos exigir más de dos reglas simultáneas.

Busquemos productos que:

- pertenezcan a Librería;
- tengan precio mayor que `1000`;
- tengan cantidad mayor o igual que `5`.


In [36]:
df.filter(
    (pl.col("categoria") == "Librería")
    & (pl.col("precio") > 1000)
    & (pl.col("cantidad") >= 5)
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Lapicera""","""Librería""",1200,10
"""Carpeta""","""Librería""",4200,5


La salida tiene `shape: (2, 4)` y conserva `Lapicera` y `Carpeta`. Ambas pertenecen a Librería, tienen precio mayor que `1000` y cantidad mayor o igual que `5`.

`Cuaderno` falla en la condición de cantidad y `Regla` en la condición de precio.

## 15. Mezclar AND y OR

Ahora construiremos una consulta más rica:

> productos de Librería cuyo precio sea mayor que `3000`, o cualquier producto de Tecnología.


In [37]:
df.filter(
    (
        (pl.col("categoria") == "Librería")
        & (pl.col("precio") > 3000)
    )
    | (pl.col("categoria") == "Tecnología")
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Calculadora""","""Tecnología""",18500,3
"""Carpeta""","""Librería""",4200,5


La salida tiene tres filas: `Cuaderno`, `Calculadora` y `Carpeta`. Los paréntesis agrupan primero `Librería AND precio > 3000`; ese bloque selecciona `Cuaderno` y `Carpeta`. Luego OR incorpora también cualquier producto de Tecnología, por lo que se suma `Calculadora`.

## 16. Cambiar los paréntesis cambia el significado

Veamos otra combinación:

```text
Librería AND (precio > 3000 OR cantidad >= 6)
```


In [38]:
df.filter(
    (pl.col("categoria") == "Librería")
    & (
        (pl.col("precio") > 3000)
        | (pl.col("cantidad") >= 6)
    )
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


La salida tiene cuatro filas: `Cuaderno`, `Lapicera`, `Regla` y `Carpeta`. Todas pertenecen a Librería y, además, cada una cumple al menos una de las dos condiciones internas: `precio > 3000` o `cantidad >= 6`.

Este resultado muestra por qué mover los paréntesis cambiaría la pregunta lógica.

## 17. Construir una consulta por partes

Cuando la lógica empieza a crecer, podemos dar nombre a cada componente.


In [39]:
es_libreria = pl.col("categoria") == "Librería"
precio_alto = pl.col("precio") > 3000
cantidad_alta = pl.col("cantidad") >= 6

condicion = (
    es_libreria
    & (precio_alto | cantidad_alta)
)

df.filter(condicion)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


La salida vuelve a contener las mismas cuatro filas: `Cuaderno`, `Lapicera`, `Regla` y `Carpeta`. Construir primero `es_libreria`, `precio_alto` y `cantidad_alta` no altera la lógica; hace que la condición final resulte más fácil de leer y mantener.

## 18. Varios predicados directamente en `filter()`

Cuando todas las condiciones deben cumplirse, Polars permite pasarlas como argumentos separados dentro de `filter()`.


In [40]:
df.filter(
    pl.col("categoria") == "Librería",
    pl.col("precio") > 1000,
    pl.col("cantidad") >= 5,
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Lapicera""","""Librería""",1200,10
"""Carpeta""","""Librería""",4200,5


La salida tiene dos filas: `Lapicera` y `Carpeta`. Los tres predicados separados dentro de `filter()` se combinaron mediante AND, por lo que ambas filas cumplen simultáneamente categoría Librería, precio mayor que `1000` y cantidad mayor o igual que `5`.

Esta forma es especialmente clara cuando todas las reglas deben cumplirse y no necesitamos OR ni agrupaciones más complejas.

## 19. Una forma compacta para igualdad

En filtros sencillos de igualdad también podemos utilizar argumentos con nombre.


In [41]:
df.filter(
    categoria="Librería"
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


La salida tiene `shape: (4, 4)` y conserva `Cuaderno`, `Lapicera`, `Regla` y `Carpeta`, exactamente las filas cuya `categoria` es `"Librería"`.

La sintaxis con argumento nombrado es compacta para igualdades simples, pero no reemplaza las expresiones cuando necesitamos comparaciones como `>`, `<` o combinaciones booleanas.

## 20. Una consulta completa

Construimos ahora dos reglas con significado propio:

- la categoría debe ser Librería o Tecnología;
- el precio debe superar `3000`.


In [42]:
categoria_interes = pl.col("categoria").is_in(
    ["Librería", "Tecnología"]
)

precio_interes = pl.col("precio") > 3000

df.filter(
    categoria_interes & precio_interes
).select(
    "producto",
    "categoria",
    "precio",
)

producto,categoria,precio
str,str,i64
"""Cuaderno""","""Librería""",3500
"""Calculadora""","""Tecnología""",18500
"""Carpeta""","""Librería""",4200


La salida final tiene `shape: (3, 3)` y contiene `Cuaderno`, `Calculadora` y `Carpeta`, mostrando únicamente `producto`, `categoria` y `precio`.

Las dos expresiones intermedias separan la lógica en bloques legibles: primero definimos qué categorías nos interesan y qué precios consideramos de interés; luego exigimos ambas condiciones con AND y finalmente reducimos las columnas mediante `select()`.

## 21. Qué conviene recordar

En este cuaderno aprendimos a construir condiciones compuestas mediante:

```text
&    AND
|    OR
~    NOT
```

Los paréntesis forman parte de la lógica de la expresión y conviene utilizarlos siempre para dejar claro cómo se agrupan las comparaciones.

También vimos que podemos:

- guardar condiciones en variables;
- observarlas como columnas booleanas antes de filtrar;
- utilizar `is_in()` cuando varias alternativas pertenecen a una misma columna;
- pasar varios predicados directamente a `filter()` cuando todos deben cumplirse;
- combinar expresiones complejas y luego encadenar `select()`.

A partir de este punto ya podemos formular filtros mucho más próximos a las preguntas reales de un análisis.


## Próximo paso

Hasta aquí utilizamos las expresiones principalmente para seleccionar columnas y decidir qué filas conservar.

En el próximo capítulo comenzaremos a utilizarlas para **crear y transformar columnas**. Allí aparecerá `with_columns()`, que nos permitirá conservar la estructura del DataFrame y agregar nuevas variables calculadas o reemplazar columnas existentes mediante expresiones.


---

### Autoría y atribución

Este cuaderno forma parte del material educativo desarrollado por **VintaBytes**.

📧 **Contacto:** [vintabytes@gmail.com](mailto:vintabytes@gmail.com)

🌐 **Repositorio oficial:** [github.com/VintaBytes/Ciencia-de-datos](https://github.com/VintaBytes/Ciencia-de-datos)

Se permite compartir y reutilizar este material respetando la autoría y manteniendo visible esta atribución.

---
